# NucleiSky2D Benchmarking

## 🎯 Objective
The goal of this notebook is to determine the **minimal number of nuclei** (or minimal patch size) required to reliably locate a cropped image within a large reference image ("The Haystack").

This quantifies the robustness of matching algorithms (Graph, Quad, Triangles, Hashing) and answers the question: *How much biological context is needed to fingerprint a specific tissue region?*

## ⚙️ Workflow
1.  **Select Reference Image**: Load a full slide or large field-of-view image.
2.  **Segmentation**: Detect nuclei and extract centroids/features from the full image.
3.  **Synthetic Benchmarking**:
    * The system generates random "virtual crops" of varying sizes from the full image.
    * It filters for crops containing specific numbers of nuclei.
    * It attempts to match these crops back to the full image using various algorithms.
4.  **Analysis**:
    * Calculate the success rate as a function of nuclei count ($N$) and patch size.
    * Fit a logistic regression (sigmoid) curve to the binary success data to determine the precise, continuous threshold where re-identification probability reaches the 90% target.
    * Evaluate computational efficiency by tracking matching speed against the number of nuclei.

## 🔑 Key Metrics
* **Unified Success**: A match is considered successful *only* if it simultaneously meets four strict criteria:
    1.  The base algorithm reports a successful match.
    2.  **Translation Error**: The estimated position is within a small margin of error (e.g., < 20 px) of the true ground-truth coordinate.
    3.  **Inlier Fraction**: At least 80% of the nuclei in the crop align mathematically with the reference map.
    4.  **SSIM (Structural Similarity)**: The matched image region achieves an SSIM of at least 0.80 compared to the ground truth.
* **$N_{min}$**: The smallest number of nuclei required to achieve a $\ge$ 90% success rate, derived from the sigmoid curve fit.
* **Execution Speed**: The total time in seconds required to compute the match, evaluating how each algorithm scales with increased biological context.

# 0. Install + import functions

In [ ]:
# @title  Initialize Environment & Install Dependencies

import sys, importlib
import torch
from IPython.display import display

# Metadata
current_version = "0.0.4"
notebook_name = "NucleiSky2DBenchmarking"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    !pip install -q "cellpose[all]" tifffile
    !pip install -q instanseg-torch

    from google.colab import userdata

    # 1. Retrieve the token securely from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print("Error: Secret 'GITHUB_TOKEN' not found. Please add it in the sidebar (🔑).")
        token = None

    # 2. Install from your private repo
    if token:
        user = 'CellMigrationLab'  # Replace with your GitHub username
        repo = 'nucleisky2d'

        # The --upgrade flag ensures you always get the latest version if you re-run
        !pip install --upgrade git+https://{token}@github.com/{user}/{repo}.git

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")

else:
    # Fallback for local environments
    print("done")


# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
    print("    -> In Colab, go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")

# ---------------------------------------------------------
# 3. Package Compatibility Layer (nucleisky2d vs nucleiskyapp)
# ---------------------------------------------------------
for pkg_name in ["nucleisky2d", "nucleisky3d"]:
    try:
        pkg = importlib.import_module(pkg_name)
        # Create a dummy namespace for the umbrella package if needed
        if "nucleiskyapp" not in sys.modules:
            sys.modules["nucleiskyapp"] = type(sys)("nucleiskyapp")
        sys.modules[f"nucleiskyapp.{pkg_name}"] = pkg
    except ModuleNotFoundError:
        pkg = importlib.import_module(f"nucleiskyapp.{pkg_name}")
        sys.modules[pkg_name] = pkg


# ---------------------------------------------------------
# 4. Main Imports
# ---------------------------------------------------------

from IPython.display import display, clear_output
from tifffile import imwrite, imread
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import html
import copy
import traceback

import os
import time
import zlib

import pandas as pd

from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim


import nucleisky2d
from nucleisky2d.demo_utils import generate_random_crop


# IO
from nucleisky2d.io import (
    load_image,
    load_transforms_any,
    get_pixel_size_um_from_tiff,
    save_json,
    make_result_dir,
    save_nucleisky_transform,
    append_transform_jsonl
)

# Preprocess
from nucleisky2d.preprocess import (
    require_2d_image,
    require_2d_label_mask,
    require_positive_float,
    scale_normalize_pair_for_segmentation,
    ij_percentile_normalize
)


# 1. Pure Export
from nucleisky2d.export import (
    export_aligned_imagej_stacks,
    run_export_with_record,
    inspect_image_header,
)

# 2. Visualization
from nucleisky2d.visualization import (
    imshow_safe,
    show_alignment_original_and_rescaled,
    downsample_points_for_display
)

# -----------------------------------------------

from nucleisky2d.segmentation import segment_nuclei_dispatch
from nucleisky2d.features import (
    extract_nuclear_features,
    add_centroids_orig_px_columns,
    centroids_from_df,
    extract_centroids_um,
    stack_feature_vectors
)
from nucleisky2d.config import save_matcher_config_json, DEFAULT_MATCHER_CONFIG
from nucleisky2d.pipeline import (
    NucleiSky,
    run_adaptive_matching_and_export,
    pick_best_transform,
    run_adaptive_nucleisky
)

print("✅ Environment Ready")

In [ ]:
#@title Benchmark functions

BENCH_COLS = [
    "matcher",
    "patch_h_px", "patch_w_px",
    "patch_y0", "patch_x0",
    "n_nuclei",
    "success",
    "frac_inliers", "mean_error_um", "trans_error_px", "ssim",
    "time_matcher_s", "time_total_s",
]

def _sanitize_controller_kwargs(d: dict | None) -> dict:
    """
    Allow ONLY keys that run_adaptive_nucleisky actually expects as controller params.
    Everything else is dropped to avoid accidentally forwarding matcher kwargs (e.g. "_common").
    """
    if d is None:
        return {}

    if not isinstance(d, dict):
        raise ValueError("For matcher='adaptive', matcher_kwargs must be a dict of controller options.")

    allowed = {
        "matcher_order",
        "base_seed",
        "matcher_config",
        "stop_on_success",
        "store_full_out",
        "max_total_time_s",
        "min_frac_inliers_success",
        "max_mean_error_um_success",
        "accept_transform_without_success",
    }

    out = {k: v for k, v in d.items() if k in allowed}
    dropped = [k for k in d.keys() if k not in allowed]
    if dropped:
        print(f"ℹ️ adaptive: dropped unsupported controller kwargs: {dropped}")
    return out



def _clip_origin(y0: int, x0: int, H: int, W: int, ph: int, pw: int) -> tuple[int, int]:
    y0 = int(max(0, min(int(y0), int(H - ph))))
    x0 = int(max(0, min(int(x0), int(W - pw))))
    return y0, x0


def _count_nuclei_in_patch(ys_px: np.ndarray, xs_px: np.ndarray, y0: int, x0: int, ph: int, pw: int) -> int:
    y1 = y0 + ph
    x1 = x0 + pw
    return int(np.sum((ys_px >= y0) & (ys_px < y1) & (xs_px >= x0) & (xs_px < x1)))


def make_patch_candidates_anchored_on_nuclei(
    H: int,
    W: int,
    patch_h_px: int,
    patch_w_px: int,
    ys_px: np.ndarray,
    xs_px: np.ndarray,
    max_trials: int,
    seed: int,
    min_nuclei: int = 1,
    oversample_factor: int = 25,
) -> np.ndarray:
    """
    Deterministically propose up to max_trials patch origins (y0,x0) such that
    each patch contains at least min_nuclei nuclei (based on ys_px/xs_px).
    """
    patch_h_px = int(patch_h_px); patch_w_px = int(patch_w_px)
    max_trials = int(max_trials)
    min_nuclei = int(min_nuclei)

    if H <= patch_h_px or W <= patch_w_px:
        return np.zeros((0, 2), dtype=int)

    ys_px = np.asarray(ys_px, dtype=int)
    xs_px = np.asarray(xs_px, dtype=int)
    if ys_px.size == 0:
        return np.zeros((0, 2), dtype=int)

    rng = np.random.default_rng(int(seed))

    # We try more proposals than needed, then keep the first max_trials that satisfy min_nuclei.
    n_proposals = max(max_trials * int(oversample_factor), max_trials)
    idx = rng.integers(0, len(ys_px), size=n_proposals, endpoint=False)

    out = []
    seen = set()

    for i in idx:
        y = int(ys_px[i])
        x = int(xs_px[i])

        # Place the chosen nucleus at a random location inside the patch
        oy = int(rng.integers(0, patch_h_px))
        ox = int(rng.integers(0, patch_w_px))

        y0, x0 = _clip_origin(y - oy, x - ox, H, W, patch_h_px, patch_w_px)

        if (y0, x0) in seen:
            continue

        if min_nuclei > 1:
            n_in = _count_nuclei_in_patch(ys_px, xs_px, y0, x0, patch_h_px, patch_w_px)
            if n_in < min_nuclei:
                continue

        seen.add((y0, x0))
        out.append((y0, x0))
        if len(out) >= max_trials:
            break

    return np.asarray(out, dtype=int)



def _stable_u32_from_str(s: str) -> int:
    """Stable across Python sessions (unlike built-in hash())."""
    return zlib.adler32(s.encode("utf-8")) & 0xFFFFFFFF


import copy

def _deep_merge_dict(base: dict, override: dict | None) -> dict:
    """Recursively merge override into base without mutating either."""
    if override is None:
        return copy.deepcopy(base)
    out = copy.deepcopy(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge_dict(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def make_patch_candidates(H, W, patch_h_px, patch_w_px, max_trials, seed, stride_frac=0.5):
    """
    Return up to max_trials candidate (y0,x0) patch origins.
    Deterministic for a given (H,W,patch_h_px,patch_w_px,seed,stride_frac).
    """
    patch_h_px = int(patch_h_px); patch_w_px = int(patch_w_px)
    if H <= patch_h_px or W <= patch_w_px:
        return np.zeros((0, 2), dtype=int)

    stride_y = max(1, int(round(patch_h_px * float(stride_frac))))
    stride_x = max(1, int(round(patch_w_px * float(stride_frac))))

    y_candidates = np.arange(0, H - patch_h_px + 1, stride_y, dtype=int)
    x_candidates = np.arange(0, W - patch_w_px + 1, stride_x, dtype=int)

    Y0, X0 = np.meshgrid(y_candidates, x_candidates, indexing="ij")
    candidates = np.column_stack([Y0.ravel(), X0.ravel()]).astype(int, copy=False)

    rng = np.random.default_rng(int(seed))
    rng.shuffle(candidates)

    if len(candidates) > int(max_trials):
        candidates = candidates[: int(max_trials)]
    return candidates


def make_benchmark_patch_plan(
    img_full,
    patch_sizes_px,
    patches_per_size,
    seed=0,
    stride_frac=0.5,   # kept for compatibility; not used in anchored mode
    df_full=None,
    min_nuclei=1,      # set to 3 to avoid your n_nuc<3 early skip
):
    """
    Returns dict: {size_px: candidates_array(N,2) of (y0,x0)}.

    If df_full is provided, candidates are anchored on nuclei and filtered to contain >= min_nuclei nuclei.
    Deterministic per (seed, size).
    """
    img_full = np.asarray(img_full)
    if img_full.ndim != 2:
        raise ValueError(f"Expected 2D image. Got shape={img_full.shape}")

    H, W = img_full.shape
    plan = {}

    # Precompute nuclei coords (px) if provided
    ys_px = xs_px = None
    if df_full is not None:
        if ("centroid_y_px" not in df_full.columns) or ("centroid_x_px" not in df_full.columns):
            raise ValueError("df_full must have centroid_y_px and centroid_x_px to enforce min_nuclei.")
        ys_px = df_full["centroid_y_px"].to_numpy(dtype=int, copy=False)
        xs_px = df_full["centroid_x_px"].to_numpy(dtype=int, copy=False)

    for size in patch_sizes_px:
        size = int(size)
        ss = np.random.SeedSequence([int(seed), int(size)])
        size_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        if df_full is not None:
            cand = make_patch_candidates_anchored_on_nuclei(
                H, W,
                patch_h_px=size,
                patch_w_px=size,
                ys_px=ys_px,
                xs_px=xs_px,
                max_trials=int(patches_per_size),
                seed=int(size_seed),
                min_nuclei=int(min_nuclei),
            )
        else:
            # fallback to your original geometric plan
            cand = make_patch_candidates(
                H, W,
                patch_h_px=size,
                patch_w_px=size,
                max_trials=int(patches_per_size),
                seed=int(size_seed),
                stride_frac=float(stride_frac),
            )

        plan[size] = cand

    return plan



def _benchmark_runtime_overrides(matcher: str, base_overrides: dict | None, run_seed: int) -> dict:
    """
    Ensure NucleiSky receives deterministic random_state=run_seed,
    regardless of user overrides.
    Accepts:
      - hierarchical: {"_common": {...}, "graph": {...}}
      - flat: {"n_iters": 50000, ...} (assumed matcher-specific)
    """
    matcher_mode = str(matcher).strip().lower()
    out = {"_common": {}, matcher_mode: {}}

    if base_overrides:
        if not isinstance(base_overrides, dict):
            raise ValueError("matcher_kwargs must be a dict (flat or hierarchical).")

        is_hier = ("_common" in base_overrides) or (matcher_mode in base_overrides)
        if is_hier:
            out = _deep_merge_dict(out, base_overrides)
        else:
            out[matcher_mode] = dict(base_overrides)

    # ALWAYS enforce the benchmark seed last
    out.setdefault("_common", {})
    out["_common"]["random_state"] = int(run_seed)
    return out


def _extract_quality_safe(out: dict) -> tuple[float, float | None]:
    q = out.get("match_quality", {}) or {}
    try:
        frac = float(q.get("frac_inliers", 0.0) or 0.0)
    except Exception:
        frac = 0.0
    err = q.get("mean_error_um", None)
    return frac, err


def run_single_patch_match_benchmark(
    img_full,
    df_full,
    PIXEL_SIZE_UM,
    patch_h_px,
    patch_w_px,
    max_trials=50,
    random_state=None,
    compute_ssim=True,
    matcher="graph",

    matcher_config=None,
    matcher_kwargs=None,

    save_path=None,
    restart=False,
    candidates=None,
    candidates_seed=None,
    stride_frac=0.5,
    ij_percentile_normalize=None,
):
    """
    Benchmarks a single matcher on a set of candidate patches.
    UPDATED: ij_percentile_normalize restored (required), do_plots removed (unexpected).
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize (used for SSIM calculation).")

    img_full = np.asarray(img_full)
    if img_full.ndim != 2:
        raise ValueError(f"Benchmark expects a 2D image. Got shape={img_full.shape}")

    H, W = img_full.shape
    patch_h_px = int(patch_h_px); patch_w_px = int(patch_w_px)

    if H <= patch_h_px or W <= patch_w_px:
        print(f"Patch {patch_h_px}x{patch_w_px} is larger than image {H}x{W}, skipping.")
        return []

    matcher_mode = str(matcher).strip().lower()

    # ---- Candidates (fixed across matchers if provided) ----
    if candidates is None:
        seed = int(candidates_seed if candidates_seed is not None else (random_state if random_state is not None else 0))
        candidates = make_patch_candidates(
            H, W,
            patch_h_px=patch_h_px,
            patch_w_px=patch_w_px,
            max_trials=int(max_trials),
            seed=seed,
            stride_frac=float(stride_frac),
        )
    candidates = np.asarray(candidates, dtype=int)
    if candidates.ndim != 2 or candidates.shape[1] != 2:
        raise ValueError(f"candidates must be (N,2) array of (y0,x0). Got {candidates.shape}")

    # ---- df requirements ----
    required_cols = ["centroid_y_um", "centroid_x_um", "centroid_y_px", "centroid_x_px"]
    missing_cols = [c for c in required_cols if c not in df_full.columns]
    if missing_cols:
        raise ValueError(f"df_full missing required columns: {missing_cols}")

    # For "graph" we require feature_vector
    if matcher_mode == "graph":
        if "feature_vector" not in df_full.columns:
            raise ValueError("df_full missing required column for graph matcher: 'feature_vector'")

    # Precompute full centroids once
    centroids_full_um = df_full[["centroid_y_um", "centroid_x_um"]].to_numpy(dtype=float, copy=False)

    # Precompute full features if available and needed
    has_feat = ("feature_vector" in df_full.columns) and (len(df_full) > 0)
    features_full = None
    if has_feat and (matcher_mode in ("graph", "adaptive")):
        features_full = np.stack(df_full["feature_vector"].to_numpy())

    # ---- checkpointing ----
    results = []
    already_done_coords = set()

    if save_path:
        save_path = str(save_path)
        if restart and os.path.exists(save_path):
            os.remove(save_path)
            print(f"Restart=True → deleted checkpoint: {save_path}")

        if (not restart) and os.path.exists(save_path):
            print(f"Resuming from checkpoint: {save_path}")
            try:
                prev_df = pd.read_csv(save_path)
                if ("patch_y0" in prev_df.columns) and ("patch_x0" in prev_df.columns):
                    for _, row in prev_df.iterrows():
                        y = row.get("patch_y0", None)
                        x = row.get("patch_x0", None)
                        if pd.notna(y) and pd.notna(x):
                            already_done_coords.add((int(y), int(x)))
                results = prev_df.to_dict(orient="records")
                print(f"Loaded {len(results)} previous rows; {len(already_done_coords)} with patch coords")
            except Exception as e:
                print(f"⚠️ Could not read checkpoint {save_path}: {e}")

    def _append_and_checkpoint(rowdict, y0, x0):
        rowdict = dict(rowdict)
        rowdict.setdefault("matcher", str(matcher_mode))
        rowdict.setdefault("patch_h_px", int(patch_h_px))
        rowdict.setdefault("patch_w_px", int(patch_w_px))
        rowdict.setdefault("patch_y0", int(y0))
        rowdict.setdefault("patch_x0", int(x0))

        for c in BENCH_COLS:
            rowdict.setdefault(c, np.nan)

        results.append(rowdict)

        if save_path:
            df_row = pd.DataFrame([rowdict], columns=BENCH_COLS)
            write_header = not os.path.exists(save_path)
            df_row.to_csv(save_path, mode="a", header=write_header, index=False)

        already_done_coords.add((int(y0), int(x0)))

    # Deterministic seed for per-run randomness
    run_seed = int(random_state if random_state is not None else 0)

    # ---- main loop ----
    for (y0, x0) in tqdm(candidates, desc=f"{matcher_mode} | {patch_h_px}x{patch_w_px} px patches"):
        y0 = int(y0); x0 = int(x0)
        if (y0, x0) in already_done_coords:
            continue

        ss = np.random.SeedSequence([int(run_seed), int(patch_h_px), int(patch_w_px), int(y0), int(x0), 999])
        patch_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        patch_start_t = time.perf_counter()

        y1 = y0 + patch_h_px
        x1 = x0 + patch_w_px
        if y1 > H or x1 > W:
            continue

        patch_img = img_full[y0:y1, x0:x1]

        mask_patch = (
            (df_full["centroid_y_px"] >= y0)
            & (df_full["centroid_y_px"] < y1)
            & (df_full["centroid_x_px"] >= x0)
            & (df_full["centroid_x_px"] < x1)
        )
        df_patch = df_full.loc[mask_patch].copy()
        n_nuc = int(len(df_patch))

        if n_nuc < 3:
            patch_total_t = time.perf_counter() - patch_start_t
            _append_and_checkpoint(dict(
                n_nuclei=n_nuc,
                success=False,
                frac_inliers=0.0,
                mean_error_um=None,
                trans_error_px=None,
                ssim=None,
                time_matcher_s=0.0,
                time_total_s=float(patch_total_t),
            ), y0, x0)
            continue

        # local coords (µm)
        y_local_px = df_patch["centroid_y_px"].to_numpy(dtype=float, copy=False) - float(y0)
        x_local_px = df_patch["centroid_x_px"].to_numpy(dtype=float, copy=False) - float(x0)
        centroids_crop_um = np.column_stack([y_local_px * float(PIXEL_SIZE_UM),
                                             x_local_px * float(PIXEL_SIZE_UM)])

        features_crop = None
        if ("feature_vector" in df_patch.columns) and (len(df_patch) > 0) and (matcher_mode in ("graph", "adaptive")):
            try:
                features_crop = np.stack(df_patch["feature_vector"].to_numpy())
            except Exception:
                features_crop = None

        # ---- call matcher ----
        t0_match = time.perf_counter()

        if matcher_mode == "adaptive":
            controller_kwargs = _sanitize_controller_kwargs(matcher_kwargs)
            controller_kwargs["base_seed"] = int(patch_seed)
            controller_kwargs.setdefault("stop_on_success", True)
            controller_kwargs.setdefault("store_full_out", False)

            out, _hist = run_adaptive_nucleisky(
                centroids_crop_um=centroids_crop_um,
                centroids_full_um=centroids_full_um,
                img_full=img_full,
                img_crop=patch_img,
                pixel_size_full_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_um=float(PIXEL_SIZE_UM),

                # --- FIXED: ADDED BACK ---
                ij_percentile_normalize=ij_percentile_normalize,

                features_crop=features_crop,
                features_full=features_full,
                df_full=df_full,
                df_crop=df_patch,
                matcher_config=matcher_config,
                save_dir=None,
                save_prefix="bench_adaptive",
                **controller_kwargs,
            )
        else:
            bench_overrides = _benchmark_runtime_overrides(matcher_mode, matcher_kwargs, patch_seed)
            out = NucleiSky(
                centroids_crop_um=centroids_crop_um,
                centroids_full_um=centroids_full_um,
                img_full=img_full,
                img_crop=patch_img,
                pixel_size_full_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_um=float(PIXEL_SIZE_UM),
                matcher=str(matcher_mode),

                # --- FIXED: ADDED BACK ---
                ij_percentile_normalize=ij_percentile_normalize,

                features_crop=(features_crop if matcher_mode == "graph" else None),
                features_full=(features_full if matcher_mode == "graph" else None),
                df_full=df_full,
                df_crop=df_patch,
                matcher_config=matcher_config,
                matcher_kwargs=bench_overrides,
                save_dir=None,
                save_prefix="bench",
            )

        time_matcher_s = time.perf_counter() - t0_match

        if out.get("best_t", None) is None:
            patch_total_t = time.perf_counter() - patch_start_t
            frac_inliers, mean_error_um = _extract_quality_safe(out)
            _append_and_checkpoint(dict(
                n_nuclei=n_nuc,
                success=False,
                frac_inliers=float(frac_inliers),
                mean_error_um=mean_error_um,
                trans_error_px=None,
                ssim=None,
                time_matcher_s=float(time_matcher_s),
                time_total_s=float(patch_total_t),
            ), y0, x0)
            continue

        best_t = np.asarray(out["best_t"], float).reshape(2,)
        q = out.get("match_quality", {}) or {}

        success_flag = bool(out.get("success", False))
        try:
            frac_inliers = float(q.get("frac_inliers", np.nan))
        except Exception:
            frac_inliers = np.nan
        mean_error_um = q.get("mean_error_um", None)

        true_topleft_um = np.array([y0 * float(PIXEL_SIZE_UM), x0 * float(PIXEL_SIZE_UM)], dtype=float)
        trans_error_px = float(np.linalg.norm(best_t - true_topleft_um) / float(PIXEL_SIZE_UM))

        ssim_val = None
        if compute_ssim:
            y_pred0 = int(round(best_t[0] / float(PIXEL_SIZE_UM)))
            x_pred0 = int(round(best_t[1] / float(PIXEL_SIZE_UM)))
            if (0 <= y_pred0 <= H - patch_h_px) and (0 <= x_pred0 <= W - patch_w_px):
                pred_roi = img_full[y_pred0:y_pred0 + patch_h_px, x_pred0:x_pred0 + patch_w_px]
                # Still using ij_percentile_normalize for SSIM
                patch_norm = ij_percentile_normalize(patch_img)
                pred_norm  = ij_percentile_normalize(pred_roi)
                ssim_val = float(ssim(patch_norm, pred_norm, data_range=1.0))

        patch_total_t = time.perf_counter() - patch_start_t

        _append_and_checkpoint(dict(
            n_nuclei=n_nuc,
            success=bool(success_flag),
            frac_inliers=frac_inliers,
            mean_error_um=mean_error_um,
            trans_error_px=trans_error_px,
            ssim=ssim_val,
            time_matcher_s=float(time_matcher_s),
            time_total_s=float(patch_total_t),
        ), y0, x0)

    return results


def benchmark_patch_sizes_on_current_image_multi(
    img_full,
    df_full,
    PIXEL_SIZE_UM,
    patch_sizes_px=(20, 50, 75, 100),
    patches_per_size=20,
    random_state=0,
    matchers=("graph", "quad", "triangles", "hashing", "adaptive"),

    matcher_config=None,
    matcher_kwargs_map=None,  # can include "adaptive": {controller kwargs...}

    save_path_dir=None,
    restart=False,
    stride_frac=0.5,
    ij_percentile_normalize=None,

    # NEW
    min_nuclei=1,   # recommend 3 if you keep the n_nuc < 3 early-skip in run_single_patch_match_benchmark
):
    """
    Runs all matchers on the SAME candidate patches for each patch size.

    If min_nuclei > 0, we anchor candidate patches on nuclei (via df_full) and
    filter to contain at least min_nuclei nuclei.
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize.")

    if matcher_kwargs_map is None:
        matcher_kwargs_map = {}

    # Patch plan built ONCE per size -> ensures same patches across matchers
    patch_plan = make_benchmark_patch_plan(
        img_full=img_full,
        patch_sizes_px=patch_sizes_px,
        patches_per_size=int(patches_per_size),
        seed=int(random_state),
        stride_frac=float(stride_frac),

        # NEW: enable anchored mode
        df_full=df_full,
        min_nuclei=int(min_nuclei),
    )

    all_results = []

    for size in patch_sizes_px:
        size = int(size)
        candidates = patch_plan[size]
        print(f"\n=== Patch size {size}×{size} px | N candidates = {len(candidates)} ===")

        for matcher in matchers:
            matcher = str(matcher).strip().lower()
            print(f"\n========== Matcher: {matcher} ==========")

            save_path = None
            if save_path_dir:
                os.makedirs(save_path_dir, exist_ok=True)
                save_path = os.path.join(save_path_dir, f"{matcher}_patch{size}px.csv")

            # deterministic per (global seed, size, matcher)
            ss_run = np.random.SeedSequence([int(random_state), int(size), _stable_u32_from_str(matcher)])
            run_seed = int(ss_run.generate_state(1, dtype=np.uint32)[0])

            res = run_single_patch_match_benchmark(
                img_full=img_full,
                df_full=df_full,
                PIXEL_SIZE_UM=float(PIXEL_SIZE_UM),
                patch_h_px=size,
                patch_w_px=size,
                max_trials=int(patches_per_size),
                random_state=int(run_seed),
                compute_ssim=True,
                matcher=matcher,
                matcher_config=matcher_config,
                matcher_kwargs=matcher_kwargs_map.get(matcher, None),
                save_path=save_path,
                restart=bool(restart),
                candidates=candidates,
                ij_percentile_normalize=ij_percentile_normalize,
            )
            all_results.extend(res)

    df_results = pd.DataFrame(all_results)
    return all_results, df_results



print("Done.")

# 1. Load your images and extract features

In [ ]:
#@title Step 1: Choose the image to benchmark
# ============================================

# ---- Imports ----
import html
import ipywidgets as widgets
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# ----------------------------
# UI Styling & CSS
# ----------------------------
STYLE_CSS = """
<style>
    .ns-card {
        border: 1px solid #E2E8F0;
        border-radius: 8px;
        padding: 16px;
        margin-bottom: 16px;
        background-color: #FFFFFF;
        box-shadow: 0 1px 3px 0 rgb(0 0 0 / 0.1);
    }
    .ns-card-warn {
        border: 1px solid #FECACA; /* Red-200 */
        background-color: #FEF2F2; /* Red-50 */
    }
    .ns-header {
        font-weight: 700;
        font-size: 14px;
        color: #0F172A;
        margin-bottom: 12px;
        text-transform: uppercase;
        letter-spacing: 0.05em;
        border-bottom: 1px solid #F1F5F9;
        padding-bottom: 8px;
    }
    .ns-label {
        font-size: 11px;
        font-weight: 700;
        color: #64748B;
        margin-top: 8px;
        margin-bottom: 4px;
        text-transform: uppercase;
    }
</style>
"""

# ----------------------------
# Utilities
# ----------------------------
def _status_ok(lines):
    safe = "<br>".join(html.escape(str(x)) for x in lines)
    return f"""
    <div style='font-size:13px; color:#065F46; background:#ECFDF5;
    border:1px solid #A7F3D0; padding:10px; border-radius:6px; line-height:1.45;'>
    {safe}</div>"""

def _status_err(lines, title="Action required"):
    safe = "<br>".join(html.escape(str(x)) for x in lines)
    return f"""
    <div style="border:1px solid #FECACA; background:#FEF2F2; border-radius:6px; padding:10px; margin:8px 0;">
      <div style="font-weight:700; color:#991B1B; font-size:13px; margin-bottom:4px;">{html.escape(title)}</div>
      <div style="font-size:12px; color:#7F1D1D; line-height:1.35;">{safe}</div>
    </div>
    """

def require_positive_float(v, label):
    v = float(v)
    if not np.isfinite(v) or v <= 0:
        raise ValueError(f"{label} must be a positive number (µm/px).")
    return v

def _desc_width(*ws, w="90px"):
    for x in ws:
        try: x.style.description_width = w
        except: pass

# ------------------------------------------------------------
# State & Globals
# ------------------------------------------------------------
state = dict(
    confirmed_full=False,
    last_full_path="",
    need_full=False,
)

# Prefill
_last_full = float(globals().get("_NUCLEISKY_LAST_MANUAL_FULL", 1.0))
_default_out_dir = str(globals().get("RESULT_DIR", ""))

# ----------------------------
# WIDGETS CONSTRUCTION
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)

# --- Header ---
title_html = widgets.HTML(
    "<div style='font-weight:900; font-size:18px; color:#0F172A; margin:0;'>Choose Image</div>"
    "<div style='font-size:13px; color:#64748B;'>Select your full reference image and output location.</div>"
)

# --- Card 1: Main Inputs ---
# Set a default value that actually exists if possible, or leave blank to force user input
img_path = widgets.Text(
    value='/content/gdrive/MyDrive/Work/manuscript/Ongoing Projects/NucleiSky/Datasets/microfluidic channel fixed/Image1.tif',
    description='Full Path', placeholder='/path/to/large_image.tif', layout=widgets.Layout(width='98%')
)

out_dir_input = widgets.Text(
    value=_default_out_dir,
    description='Output Dir',
    placeholder='(Optional) Leave empty for auto-timestamp',
    layout=widgets.Layout(width='98%')
)

input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>1. Configuration</div>"),
    img_path,
    widgets.HTML("<div style='height:8px'></div>"),
    widgets.HTML("<div class='ns-label'>Results Directory</div>"),
    out_dir_input,
    widgets.HTML("<div style='font-size:11px; color:#94A3B8; margin-top:2px;'>If left blank, a new timestamped folder will be created automatically.</div>")
])
input_card.add_class("ns-card")

# --- Card 2: Manual Metadata (Warning Card) ---
#
manual_msg = widgets.HTML("")
manual_full = widgets.FloatText(value=_last_full, description='Full µm/px')

manual_card = widgets.VBox([
    widgets.HTML("<div class='ns-header' style='color:#991B1B; border-color:#FECACA;'>Missing Metadata</div>"),
    manual_msg,
    widgets.HBox([manual_full], layout=widgets.Layout(width="100%", margin="5px 0")),
])
manual_card.add_class("ns-card")
manual_card.add_class("ns-card-warn")
manual_card.layout.display = "none"

# --- Outputs ---
status_html = widgets.HTML("")
run_button = widgets.Button(description=' Load & Process', button_style='primary', icon='play', layout=widgets.Layout(width='100%', height='40px'))
fig_out = widgets.Output(layout=widgets.Layout(max_height="500px", overflow="auto"))

# Styling widths
_desc_width(img_path, out_dir_input, w="80px")
_desc_width(manual_full, w="90px")

# ----------------------------
# Logic & Sync
# ----------------------------
def _sync_ui():
    show_manual = state["need_full"]
    manual_card.layout.display = "flex" if show_manual else "none"
    manual_full.layout.visibility = "visible" if state["need_full"] else "hidden"

def _on_paths_changed(_=None):
    fp = img_path.value.strip()

    if fp != state["last_full_path"]:
        state["confirmed_full"] = False
        state["last_full_path"] = fp

    state["need_full"] = False
    manual_msg.value = ""
    _sync_ui()

img_path.observe(_on_paths_changed, names="value")

# ----------------------------
# Main Handler
# ----------------------------
def on_run_clicked(b):
    # Use output widget context immediately to capture print statements
    with fig_out:
        clear_output(wait=True)
        print("Starting processing...") # Immediate feedback

        try:
            global img_full, PIXEL_SIZE_UM, RESULT_DIR
            global _NUCLEISKY_LAST_MANUAL_FULL

            status_html.value = "<div style='color:#2563EB; font-weight:600;'>Processing...</div>"

            _on_paths_changed()
            # 1. Get and clean the path
            full_path_str = img_path.value.strip()
            full_path = Path(full_path_str)
            custom_out = out_dir_input.value.strip()

            # 2. Debug Print to help you see what the computer sees
            if not full_path.exists():
                print(f"❌ ERROR: Could not find file at: '{full_path_str}'")
                print(f"   Current working directory: {Path.cwd()}")
                raise FileNotFoundError(f"Full image not found at: {full_path_str}")
            else:
                print(f"✅ Found image at: {full_path_str}")

            # Load Image
            # Ensure these functions exist in your global scope!
            if 'load_image' not in globals():
                 raise NameError("Function 'load_image' is not defined. Run the setup cells first.")

            full_raw = load_image(str(full_path))
            img_full_2d = require_2d_image(full_raw, label="Full image")

            # Check Pixel Metadata
            missing = []
            pix_full_meta = get_pixel_size_um_from_tiff(str(full_path))

            if pix_full_meta is None:
                if state["confirmed_full"]:
                    pix_full = require_positive_float(manual_full.value, "Full µm/px")
                    pix_full_src = "manual"
                else:
                    missing.append("Full Image")
                    state["need_full"] = True
            else:
                pix_full = float(pix_full_meta)
                pix_full_src = "metadata"
                state["need_full"] = False
                state["confirmed_full"] = False

            if missing:
                if state["need_full"]: state["confirmed_full"] = True
                manual_msg.value = f"<b>Pixel size not found:</b> {', '.join(missing)}."
                status_html.value = ""
                _sync_ui()
                print("⚠️ Missing metadata. Please fill in the red box that appeared.")
                return

            # Handle Output Directory
            if custom_out:
                p_out = Path(custom_out)
                p_out.mkdir(parents=True, exist_ok=True)
                RESULT_DIR = p_out
                print(f"Using custom output directory: {RESULT_DIR}")
            else:
                # Auto-generate timestamped folder
                RESULT_DIR = make_result_dir(big_image_path=str(full_path), tag="NucleiSky")

            # Ensure RESULT_DIR is a Path object for operations below
            RESULT_DIR = Path(RESULT_DIR)

            _NUCLEISKY_LAST_MANUAL_FULL = float(manual_full.value)
            globals()["_NUCLEISKY_LAST_MANUAL_FULL"] = _NUCLEISKY_LAST_MANUAL_FULL

            state["need_full"] = False
            manual_msg.value = ""
            _sync_ui()

            img_full = img_full_2d
            PIXEL_SIZE_UM = float(pix_full)

            # Save inputs
            save_json(RESULT_DIR / "inputs_metadata.json", {
                "full_image_path": str(full_path),
                "pixel_size_full": PIXEL_SIZE_UM,
                "result_dir": str(RESULT_DIR)
            })

            status_html.value = _status_ok([
                f"<b>Success!</b> Working Directory: {RESULT_DIR.name}",
                f"Path: {str(RESULT_DIR)}",
                f"Full: {PIXEL_SIZE_UM:.4f} µm/px ({pix_full_src})",
            ])

            # Display Image
            fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=100)
            imshow_safe(ax, img_full_2d, title="Full Image", max_dim=2500)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            import traceback
            traceback.print_exc()
            status_html.value = _status_err([str(e)], title="Error during processing")

run_button.on_click(on_run_clicked)

# ----------------------------
# Final Display
# ----------------------------
ui = widgets.VBox([
    style_html,
    title_html,
    input_card,
    manual_card,
    widgets.HBox([status_html], layout=widgets.Layout(margin="10px 0")),
    run_button,
    fig_out
], layout=widgets.Layout(width="90%", margin="0 auto"))

_sync_ui()
display(ui)

In [ ]:
#@title Step 2: Segmentation and Feature Extraction

# ---- Imports ----
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np
import traceback
import matplotlib.pyplot as plt

# ----------------------------
# UI Styling
# ----------------------------
STYLE_CSS = """
<style>
    .ns-card {
        border: 1px solid #E2E8F0; border-radius: 8px; padding: 16px;
        margin-bottom: 16px; background-color: #FFFFFF;
        box-shadow: 0 1px 3px 0 rgb(0 0 0 / 0.1);
    }
    .ns-header {
        font-weight: 700; font-size: 14px; color: #0F172A;
        margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.05em;
        border-bottom: 1px solid #F1F5F9; padding-bottom: 8px;
    }
    .ns-label { font-size: 12px; font-weight: 600; color: #475569; margin-bottom: 4px; }
    .ns-param-row { display: flex; gap: 12px; align-items: center; margin-bottom: 8px; }
</style>
"""
display(widgets.HTML(STYLE_CSS))

# ----------------------------
# WIDGETS
# ----------------------------

# --- Card 1: Source Selection ---
source_mode = widgets.ToggleButtons(
    options=[('Run New Segmentation', 'new'), ('Load Existing Mask', 'existing')],
    value='new',
    description='',
    button_style='',
    tooltips=['Run algorithms like Cellpose or InstanSeg', 'Load pre-computed binary TIFF mask'],
    layout=widgets.Layout(width='auto')
)

# --- Path Inputs (For Existing Masks) ---
mask_path_full = widgets.Text(description="Mask Path", placeholder="/path/to/mask_full.tif", layout=widgets.Layout(width="98%"))

mask_input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Mask Path</div>"),
    mask_path_full
])
mask_input_card.layout.display = "none" # Hidden by default

# --- Card 2: Segmentation Method ---
seg_method = widgets.Dropdown(
    options=[
        ("Cellpose (Deep Learning)", "cellpose"),
        ("InstanSeg (Fast Deep Learning)", "instanseg"),
        ("Auto-threshold (Classic CV)", "threshold"),
    ],
    value="cellpose",
    layout=widgets.Layout(width="98%")
)

method_desc = widgets.HTML("") # Dynamic description

# ---- InstanSeg Specifics ----
inst_model = widgets.Dropdown(options=["brightfield_nuclei", "fluorescence_nuclei_and_cells"], value="brightfield_nuclei", description="Model")
inst_target = widgets.Dropdown(options=["nuclei", "cells"], value="nuclei", description="Target")

# Advanced InstanSeg
inst_mode = widgets.Dropdown(options=[("Auto (small/medium)", "auto"), ("Small image", "small"), ("Medium (tiled)", "medium")], value="auto", description="Mode")
inst_clean = widgets.Checkbox(value=True, description="Cleanup Fragments")
inst_px_ovr = widgets.Checkbox(value=False, description="Override µm/px")
inst_px_val = widgets.FloatText(value=0.5, description="Value", layout=widgets.Layout(width="150px"), disabled=True)
inst_verb = widgets.IntSlider(value=0, min=0, max=2, description="Verbosity")

inst_adv_ui = widgets.VBox([
    inst_mode,
    widgets.HBox([inst_px_ovr, inst_px_val]),
    inst_clean,
    inst_verb
])
inst_accord = widgets.Accordion(children=[inst_adv_ui], titles=('Advanced InstanSeg Settings',))

inst_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Model Configuration</div>"),
    widgets.HBox([inst_model, inst_target], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    inst_accord
])

# ---- Threshold Specifics ----
thr_method = widgets.Dropdown(options=["otsu", "li", "yen", "triangle", "isodata"], value="otsu", description="Algorithm")
thr_channel = widgets.IntText(value=0, description="Channel Index", layout=widgets.Layout(width="180px"))

# Advanced Threshold
thr_sigma = widgets.FloatSlider(value=1.0, min=0, max=5, step=0.25, description="Blur Sigma")
thr_min_obj = widgets.IntText(value=80, description="Min Area (px)")
thr_watershed = widgets.Checkbox(value=True, description="Watershed Split")
thr_peak = widgets.IntText(value=5, description="Peak Dist")

thr_adv_ui = widgets.VBox([
    thr_sigma,
    widgets.HBox([thr_min_obj, thr_watershed, thr_peak])
])
thr_accord = widgets.Accordion(children=[thr_adv_ui], titles=('Advanced CV Parameters',))

thr_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Thresholding Configuration</div>"),
    widgets.HBox([thr_method, thr_channel], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    thr_accord
])

# Container for all method panels
method_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Segmentation Strategy</div>"),
    seg_method,
    widgets.HTML("<div style='height:12px'></div>"),
    method_desc,
    inst_panel,
    thr_panel
])
method_card.add_class("ns-card")

# --- Card 3: Execution ---
seg_btn = widgets.Button(
    description=" Run Segmentation",
    button_style="primary",
    icon="cogs",
    layout=widgets.Layout(width="100%", height="45px")
)

seg_out = widgets.Output(layout=widgets.Layout(max_height="600px", overflow="auto"))

# ----------------------------
# LOGIC & BINDINGS
# ----------------------------

def _update_ui_state(_=None):
    is_existing = (source_mode.value == 'existing')

    # Toggle Cards
    mask_input_card.layout.display = "flex" if is_existing else "none"
    method_card.layout.display = "none" if is_existing else "flex"

    # Toggle Method Panels
    m = seg_method.value
    inst_panel.layout.display = "flex" if m == "instanseg" else "none"
    thr_panel.layout.display = "flex" if m == "threshold" else "none"

    # Dynamic Description & Image Tags
    if m == "cellpose":
        method_desc.value = "<i>Standard deep learning model for cellular segmentation. Good general purpose choice.</i>"
    elif m == "instanseg":
        method_desc.value = "<i>High-speed pytorch implementation. Best for large datasets.</i>"
    elif m == "threshold":
        method_desc.value = "<i>Classical computer vision. Fast, but requires high contrast and clean images.</i>"

    # Button Text
    seg_btn.description = " Load Mask & Extract Features" if is_existing else " Run Segmentation & Extraction"

# InstanSeg Override logic
def _toggle_inst_px(_=None):
    inst_px_val.disabled = not inst_px_ovr.value
inst_px_ovr.observe(_toggle_inst_px, names="value")

source_mode.observe(_update_ui_state, names="value")
seg_method.observe(_update_ui_state, names="value")

# Initialize
_update_ui_state()

# ----------------------------
# EXECUTION HANDLER
# ----------------------------
def on_segment_click(_):
    # Mapping global variables to new UI
    global df_full, masks_full
    global img_full_seg
    global PIXEL_SIZE_UM_FULL_SEG

    seg_out.clear_output(wait=True)

    with seg_out:
        try:
            # 1. Pre-flight Checks
            required = ("img_full", "PIXEL_SIZE_UM")
            missing = [k for k in required if k not in globals()]
            if missing: raise RuntimeError(f"Missing Step 1 globals: {missing}")

            # 2. Get Data
            img_full_orig = require_2d_image(np.asarray(img_full), label="Full Image")
            pix_full_orig = float(globals()["PIXEL_SIZE_UM"])

            # 3. Setup Directories
            RESULT_DIR_LOCAL = Path(globals().get("RESULT_DIR", make_result_dir(tag="NucleiSky")))
            SEG_DIR = RESULT_DIR_LOCAL / "segmentation"
            SEG_DIR.mkdir(parents=True, exist_ok=True)

            # 4. Processing Mode Logic
            use_masks = (source_mode.value == 'existing')

            # We process directly on the full image resolution
            # We assume segmentation functions handle internal resizing if pixel_size_um is passed
            img_full_seg = img_full_orig
            PIXEL_SIZE_UM_FULL_SEG = pix_full_orig

            # Factor is 1.0 because we are using the original image directly
            RESCALE_FACTOR_FULL = 1.0

            if use_masks:
                print(f"📂 Loading existing mask...")

                if not mask_path_full.value.strip(): raise ValueError("Mask path required")

                masks_full = require_2d_label_mask(load_image(mask_path_full.value.strip()), label="Full Mask", expected_shape=img_full_orig.shape)

            else:
                # PERFORM SEGMENTATION
                print(f"🧠 Running {seg_method.value}...")

                # Build Settings Dictionary
                settings = {"method": seg_method.value}

                # Override logic
                pix_dispatch_full = float(PIXEL_SIZE_UM_FULL_SEG)

                if seg_method.value == "instanseg":
                    if inst_px_ovr.value:
                        pix_dispatch_full = float(inst_px_val.value)

                    settings["instanseg"] = {
                        "model_name": inst_model.value, "target": inst_target.value,
                        "mode": inst_mode.value, "verbosity": inst_verb.value,
                        "pixel_size_um": float(pix_dispatch_full),
                        "cleanup_fragments": inst_clean.value
                    }
                elif seg_method.value == "threshold":
                    #
                    settings["threshold"] = {
                        "threshold_method": thr_method.value, "channel": thr_channel.value,
                        "gaussian_sigma": thr_sigma.value, "min_object_size": thr_min_obj.value,
                        "do_watershed": thr_watershed.value, "peak_min_distance": thr_peak.value
                    }

                masks_full = segment_nuclei_dispatch(img_full_seg, seg_method.value, pix_dispatch_full, settings)

            # 5. Feature Extraction (Common)
            print("📊 Extracting morphology features...")
            df_full = extract_nuclear_features(masks_full, None, float(PIXEL_SIZE_UM_FULL_SEG), k_neighbors=10)

            # Map back to original scale (even if scale is 1.0, this ensures the correct columns 'centroid_x_px', etc. are created)
            if 'add_centroids_orig_px_columns' in globals():
                df_full = add_centroids_orig_px_columns(df_full, float(RESCALE_FACTOR_FULL))

            # 6. Saving Data
            print(f"✅ Extracted: {len(df_full)} nuclei")
            df_full.to_csv(SEG_DIR / "features_full.csv", index=False)

            # --- Save Masks with ZLIB Compression ---

            print("💾 Saving compressed segmentation mask to disk...")
            imwrite(
                SEG_DIR / "mask_full.tif",
                masks_full.astype(np.int32),
                compression="zlib"
            )

            # 7. Visualization
            fig, ax = plt.subplots(1, 1, figsize=(8, 8))

            # --- Safe Visualization ---
            disp_full = imshow_safe(ax, img_full_orig, title=f"Segmentation ({len(df_full)})")

            # Calculate coordinate scale (Display Shape / Original Shape)
            scale_y_f, scale_x_f = disp_full.shape[0] / img_full_orig.shape[0], disp_full.shape[1] / img_full_orig.shape[1]

            # Plot centroids if available
            if 'centroids_from_df' in globals():
                cent_f = centroids_from_df(df_full, use_um=False, use_orig_px=True)
                if len(cent_f):
                    # Downsample points to avoid UI crashes
                    cent_f_safe = downsample_points_for_display(cent_f)
                    # Apply scale to align points with the downsampled image axes
                    ax.scatter(cent_f_safe[:,1] * scale_x_f, cent_f_safe[:,0] * scale_y_f, s=1, c='r', alpha=0.5)

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print("❌ Error:")
            traceback.print_exc()

seg_btn.on_click(on_segment_click)

# ----------------------------
# UI Layout
# ----------------------------
ui = widgets.VBox([
    widgets.HTML("<div class='ns-header'>1. Source</div>"),
    source_mode,
    widgets.HTML("<div style='height:16px'></div>"),
    mask_input_card,
    method_card,
    widgets.HTML("<div style='height:16px'></div>"),
    seg_btn,
    seg_out
], layout=widgets.Layout(width="90%"))

display(ui)

# 2. Benchmark all matches vs one another using patches

In [ ]:
#@title Run the Benchmark

# ---------------------------------------------------------
# DYNAMIC PATH SETUP
# ---------------------------------------------------------
# Alternative 1: Use the RESULT_DIR from Step 1
if "RESULT_DIR" in globals() and RESULT_DIR:
    base_output_dir = Path(RESULT_DIR)
else:
    # Alternative 2: Fallback to local directory if pipeline wasn't run linearly
    base_output_dir = Path("benchmark_results")

# Create a dedicated subfolder for benchmarks to keep things clean
output_dir = base_output_dir / "benchmarking"
output_dir.mkdir(parents=True, exist_ok=True)

# Try to get the original filename from the metadata saved in Step 1
base_name = "current_image"
try:
    meta_path = base_output_dir / "inputs_metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)
            if "full_image_path" in meta:
                base_name = Path(meta["full_image_path"]).stem
except Exception:
    pass

print(f"📂 Saving benchmark results to: {output_dir}")
print(f"🏷️  Base filename: {base_name}")

# ---------------------------------------------------------
# PREPARE DATA
# ---------------------------------------------------------
# Use segmentation-scale image + pixel size if available
img_for_bench = globals().get("img_full_seg", globals().get("img_full"))
pix_for_bench = float(globals().get("PIXEL_SIZE_UM_FULL_SEG", globals().get("PIXEL_SIZE_UM")))

if img_for_bench is None:
    raise RuntimeError("Image data not found in globals. Please run Step 1 & 2 first.")

print("\nBenchmark settings:")
print(f"  Image shape: {np.asarray(img_for_bench).shape}")
print(f"  Scale:       {pix_for_bench:.4f} µm/px")

# ---------------------------------------------------------
# RUN BENCHMARK
# ---------------------------------------------------------
all_results, df_results = benchmark_patch_sizes_on_current_image_multi(
    img_full=img_for_bench,
    df_full=df_full,
    PIXEL_SIZE_UM=pix_for_bench,
    # Standard grid sizes
    patch_sizes_px=[50, 75, 100, 128, 192, 256, 320, 384, 448, 512],
    patches_per_size=50,
    random_state=0,
    matchers=("graph", "quad", "triangles", "hashing", "adaptive"),

    matcher_config=None,
    matcher_kwargs_map={
        "adaptive": {
            "mode": "balanced",
            "max_rounds": 2,
            # We put graph first in benchmark to see if it works optimally
            "matcher_order": ["graph", "triangles", "quad", "hashing"],
            "max_total_time_s": None,
        }
    },

    min_nuclei=3,

    # Dynamic checkpoint path
    save_path_dir=str(output_dir / f"{base_name}_checkpoints"),
    restart=False,
    ij_percentile_normalize=ij_percentile_normalize,
)

# ---------------------------------------------------------
# SAVE RESULTS
# ---------------------------------------------------------
csv_path = output_dir / f"{base_name}_benchmark_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"\n✅ Saved final benchmark CSV to:\n   {csv_path}")

# 3. Plot Results

In [ ]:
#@title reload result files if needed

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Check if df_results is already in the global environment
if 'df_results' not in globals():
    print("⚠️ 'df_results' not found in environment. Please provide the path to your CSV.")

    # Create the text input widget
    path_input = widgets.Text(
        value='',
        placeholder='e.g., /path/to/your/results.csv',
        description='CSV Path:',
        layout=widgets.Layout(width='50%')
    )

    # Create the load button
    load_button = widgets.Button(
        description='Load CSV',
        button_style='info',
        tooltip='Click to load the dataframe'
    )

    # Create an output area for success/error messages
    output_area = widgets.Output()

    def on_button_clicked(b):
        with output_area:
            output_area.clear_output()
            csv_path = path_input.value.strip()
            if not csv_path:
                print("❌ Please enter a valid path.")
                return

            try:
                # Declare global so it becomes available to the rest of the notebook
                global df_results
                df_results = pd.read_csv(csv_path)
                print(f"✅ Successfully loaded {len(df_results)} rows from '{csv_path}' into df_results!")
            except Exception as e:
                print(f"❌ Error loading file: {e}")

    load_button.on_click(on_button_clicked)

    # Display the widgets together
    display(widgets.HBox([path_input, load_button]), output_area)

else:
    print("✅ 'df_results' is already in the environment. Skipping file load.")

In [ ]:
#@title Plots


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.optimize import curve_fit
from pathlib import Path

# ==========================================
# 0. Plotting Configuration for Editable PDFs
# ==========================================
# This ensures fonts are exported as text objects (editable in Illustrator/Inkscape)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
# Optional: Set a clean, universal font
mpl.rcParams['font.family'] = 'sans-serif'

# ==========================================
# 1. Configuration & Data Prep
# ==========================================
TARGET_PROB = 0.90
FRAC_INLIERS_THRESH = 0.80
SSIM_THRESH = 0.80
ERROR_PX_THRESH = 20

def sanitize_and_score(df_results):
    df = pd.DataFrame(df_results).copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join([str(x) for x in t if str(x) != ""]).strip() for t in df.columns]
    df.columns = [str(c).strip() for c in df.columns]

    for col in ["n_nuclei", "patch_h_px", "frac_inliers", "ssim", "time_total_s"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["n_nuclei"] = df["n_nuclei"].fillna(0).astype(int)
    df["success"] = df["success"].fillna(False).astype(bool)

    df["is_success"] = (
        (df["success"] == True) &
        (df["frac_inliers"] >= FRAC_INLIERS_THRESH) &
        (df["ssim"] >= SSIM_THRESH) &
        (df.get("trans_error_px", 0) < ERROR_PX_THRESH)
    )
    return df

def wilson_interval(k, n, z=1.96):
    if n <= 0: return 0.0, 0.0
    p_hat = k / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    hw = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, center - hw), min(1.0, center + hw)

# ==========================================
# 2. Mathematical Modeling
# ==========================================
def sigmoid(x, k, x0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))

def get_min_n_for_target(k, x0, target=0.90):
    if k <= 0: return np.nan
    return x0 - np.log((1 / target) - 1) / k

# ==========================================
# 3. Plotting Functions
# ==========================================
def plot_success_and_calc_min_n(df, target=0.90, save_path=None):
    plt.figure(figsize=(10, 6), dpi=120)
    colors = sns.color_palette("husl", len(df['matcher'].unique()))
    summary_stats = []

    for i, (matcher, grp) in enumerate(df.groupby('matcher')):
        bins = np.arange(0, grp['n_nuclei'].max() + 5, 2)
        grp = grp.copy()
        grp['bin'] = pd.cut(grp['n_nuclei'], bins)
        binned = grp.groupby('bin', observed=True).agg({'is_success': 'mean', 'n_nuclei': 'mean'}).dropna()

        plt.scatter(binned['n_nuclei'], binned['is_success'], color=colors[i], alpha=0.4, s=30, label=f"{matcher} (binned data)")

        try:
            x_data = grp['n_nuclei'].values
            y_data = grp['is_success'].values.astype(float)

            popt, _ = curve_fit(sigmoid, x_data, y_data, p0=[0.5, 10], maxfev=5000)

            x_range = np.linspace(0, x_data.max(), 100)
            y_fit = sigmoid(x_range, *popt)
            plt.plot(x_range, y_fit, color=colors[i], linewidth=2.5, label=f"{matcher} (fit)")

            n_min = get_min_n_for_target(popt[0], popt[1], target)
            summary_stats.append({'matcher': matcher, f'min_nuclei_{int(target*100)}': max(1, n_min) if pd.notna(n_min) else np.nan})

        except Exception as e:
            print(f"Curve fit failed for {matcher}: {e}")
            summary_stats.append({'matcher': matcher, f'min_nuclei_{int(target*100)}': np.nan})

    plt.axhline(target, color='gray', linestyle='--', alpha=0.8, label=f'{int(target*100)}% Success Threshold')
    plt.xlabel("Number of Nuclei in Crop")
    plt.ylabel("Success Rate (Probability)")
    plt.title("Success vs Number of Nuclei (Sigmoid Fit)")

    # Place legend outside plot area safely
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.05, 1.05)

    if save_path:
        # bbox_inches='tight' prevents the legend from being chopped off
        plt.savefig(save_path, format='pdf', bbox_inches='tight', transparent=True)
        print(f"Saved: {save_path}")

    plt.show()
    return pd.DataFrame(summary_stats).sort_values(f'min_nuclei_{int(target*100)}')

def plot_success_vs_patch_size(df, save_path=None):
    plt.figure(figsize=(10, 6), dpi=120)

    for matcher, g in df.groupby(["matcher", "patch_h_px"], observed=True).agg(
        n_patches=('is_success', 'count'),
        k_success=('is_success', 'sum')
    ).reset_index().groupby('matcher'):

        g = g.sort_values('patch_h_px')
        x = g['patch_h_px'].values
        y = g['k_success'] / g['n_patches']

        bounds = [wilson_interval(k, n) for k, n in zip(g['k_success'], g['n_patches'])]
        lo = [b[0] for b in bounds]
        hi = [b[1] for b in bounds]

        p = plt.plot(x, y, "o-", linewidth=2.5, markersize=7, label=matcher)
        plt.fill_between(x, lo, hi, color=p[0].get_color(), alpha=0.15)

    plt.xlabel("Patch Size (px)")
    plt.ylabel("Success Probability")
    plt.title("Success vs Patch Size")
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.05, 1.05)

    if save_path:
        plt.savefig(save_path, format='pdf', bbox_inches='tight', transparent=True)
        print(f"Saved: {save_path}")

    plt.show()

def plot_speed_vs_nuclei(df, save_path=None):
    plt.figure(figsize=(10, 6), dpi=120)

    for matcher, g in df.groupby("matcher"):
        bins = np.arange(0, g['n_nuclei'].max() + 5, 5)
        g = g.copy()
        g['bin_mid'] = pd.cut(g['n_nuclei'], bins).apply(lambda x: x.mid if pd.notna(x) else np.nan)

        g_sorted = g.groupby("bin_mid", observed=True)["time_total_s"].mean().reset_index()
        plt.plot(g_sorted["bin_mid"], g_sorted["time_total_s"], "o-", linewidth=2, label=matcher)

    plt.xlabel("Number of Nuclei in Patch")
    plt.ylabel("Total Time [s]")
    plt.title("Speed vs Nuclei Count")
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.grid(True, alpha=0.3)

    if save_path:
        plt.savefig(save_path, format='pdf', bbox_inches='tight', transparent=True)
        print(f"Saved: {save_path}")

    plt.show()

# ==========================================
# 4. Execution Flow
# ==========================================
# Assuming df_results is already loaded in your environment
df_clean = sanitize_and_score(df_results)

# Create an output directory for the PDFs
output_dir = Path("pdf_plots")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Total evaluated patches: {len(df_clean)}")
print(f"Success Criteria: Inliers >= {FRAC_INLIERS_THRESH}, SSIM >= {SSIM_THRESH}, Error < {ERROR_PX_THRESH}px\n")

# Plot 1: Success vs Nuclei
min_req_table = plot_success_and_calc_min_n(
    df_clean,
    target=TARGET_PROB,
    save_path=output_dir / "success_vs_nuclei.pdf"
)

print(f"\n🏆 Minimal Nuclei Count for {int(TARGET_PROB*100)}% Reliability:")
display(min_req_table.style.format({f'min_nuclei_{int(TARGET_PROB*100)}': "{:.1f}"}).background_gradient(cmap='Greens'))

# Plot 2: Success vs Patch Size
plot_success_vs_patch_size(
    df_clean,
    save_path=output_dir / "success_vs_patch_size.pdf"
)

# Plot 3: Speed vs Nuclei
plot_speed_vs_nuclei(
    df_clean,
    save_path=output_dir / "speed_vs_nuclei.pdf"
)